In [204]:
import requests as r
from utils import query_raven
import pandas as pd

In [205]:

def needs_quotes(field):
    # List of fields that need single quotes
    quoted_fields = ['CustomerID', 'ShipName', 'ShipCountry', 'OrderDate', 'ShipCity']
    print(field)
    return field in quoted_fields


def format_value(parameter, value):
    if needs_quotes(parameter):
        return f"'{value}'"
    return value

def get_one_parameter_and_value(parameter1: str, operator1: str, value1: str):
    value1 = format_value(parameter1, value1)
    url = f"https://services.odata.org/V4/Northwind/Northwind.svc/Orders?$filter={parameter1} {operator1} {value1}&$format=json"
    print("Generated URL for one parameter:", url)  # Debugging print statement
    response = r.get(url)
    return response.json()

def get_two_parameters_and_values(parameter1: str, operator1: str, value1: str, parameter2: str, operator2: str, value2: str):
    value1 = format_value(parameter1, value1)
    value2 = format_value(parameter2, value2)
    url = f"https://services.odata.org/V4/Northwind/Northwind.svc/Orders?$filter={parameter1} {operator1} {value1} and {parameter2} {operator2} {value2}&$format=json"
    print("Generated URL for two parameters:", url)  # Debugging print statement
    response = r.get(url)
    return response.json()

def get_three_parameters_and_values(parameter1: str, operator1: str, value1: str, parameter2: str, operator2: str, value2: str, parameter3: str, operator3: str, value3: str):
    value1 = format_value(parameter1, value1)
    value2 = format_value(parameter2, value2)
    value3 = format_value(parameter3, value3)
    url = f"https://services.odata.org/V4/Northwind/Northwind.svc/Orders?$filter={parameter1} {operator1} {value1} and {parameter2} {operator2} {value2} and {parameter3} {operator3} {value3}&$format=json"
    print("Generated URL for three parameters:", url)  # Debugging print statement
    response = r.get(url)
    return response.json()


In [206]:
# USER_QUERY = 'how many orders were sent to Germany were delayed in their delivery'
# USER_QUERY = "How many orders sent to germany have freight cost greater than 600 $"
# USER_QUERY = "How many orders have been handled by employeeID 2. How many of those have been delayed in their delivery?"
# USER_QUERY = 'Which orders to Italy have ben processed by employee ID 4'
USER_QUERY = 'Shipping Partner Ernst Handel has done how many deliveries to Austria. how many of those orders have been delayed in their delivery ?'


In [207]:
raven_functions = \
f'''

Select the appropriate function depending on the number of parameters to be captured.

Never send null in values. If there is no value for any parameter, use a function with lesser parameters.



Operator values will be ge for Greater than or equal to
                        le for Less than or equal to
                        eq for equal to

Use this Function to pass one parameter and its value
def get_one_parameter_and_value(parameter1 : str, operator1: str , value1 : str):

Use this Function to pass two parameters and its values
def get_two_parameters_and_values(parameter1 : str, , operator1: str, value1 : str, parameter2 : str, , operator2: str value2 : str):

Use this Function to pass three parameters and its values
def get_three_parameters_and_values(parameter1 : str, , operator1: str, value1 : str, parameter2 : str, , operator2: str, , operator3: str, value2 : str, parameter3 : str, value3 : str):

The category variable can fall under any of these given below. Depending on the question asked you need to extaract relevant categories to be passed on to the API.

    **OrderID:** The unique ID of the order.
    **CustomerID:** The ID of the customer who placed the order.
    **EmployeeID:** The ID of the employee who took the order.
    **OrderDate:** The date the order was placed.
    **RequiredDate:** The date the order is required to be shipped by. RequiredDate will be less than ShippedDate for delayed deliveries.
    **ShippedDate:** The date the order was shipped. RequiredDate will be less than ShippedDate for delayed deliveries.
    **ShipVia:** The ID of the shipper used for the order. 
    **Freight:** The shipping cost of the order.
    **ShipName:** The name of the shipping recipient. This filed can be addressed as shipping partner or Shipper.
    **ShipAddress:** The address of the shipping recipient.
    **ShipCity:** The city of the shipping recipient.
    **ShipRegion:** The region of the shipping recipient (e.g., state or province).
    **ShipPostalCode:** The postal code of the shipping recipient.
    **ShipCountry:** The country of the shipping recipient.
"""

User Query: {USER_QUERY}<human_end>
    """
'''

call = query_raven(raven_functions)
print(call)

get_three_parameters_and_values(parameter1='ShipName', operator1='eq', value1='Ernst Handel', parameter2='ShipCountry', operator2='eq', value2='Austria', parameter3='ShippedDate', operator3='le', value3='RequiredDate')


In [208]:
response = eval(call)

ShipName
ShipCountry
ShippedDate
Generated URL for three parameters: https://services.odata.org/V4/Northwind/Northwind.svc/Orders?$filter=ShipName eq 'Ernst Handel' and ShipCountry eq 'Austria' and ShippedDate le RequiredDate&$format=json


In [209]:
print(response)

{'@odata.context': 'https://services.odata.org/V4/Northwind/Northwind.svc/$metadata#Orders', 'value': [{'OrderID': 10258, 'CustomerID': 'ERNSH', 'EmployeeID': 1, 'OrderDate': '1996-07-17T00:00:00Z', 'RequiredDate': '1996-08-14T00:00:00Z', 'ShippedDate': '1996-07-23T00:00:00Z', 'ShipVia': 1, 'Freight': 140.51, 'ShipName': 'Ernst Handel', 'ShipAddress': 'Kirchgasse 6', 'ShipCity': 'Graz', 'ShipRegion': None, 'ShipPostalCode': '8010', 'ShipCountry': 'Austria'}, {'OrderID': 10263, 'CustomerID': 'ERNSH', 'EmployeeID': 9, 'OrderDate': '1996-07-23T00:00:00Z', 'RequiredDate': '1996-08-20T00:00:00Z', 'ShippedDate': '1996-07-31T00:00:00Z', 'ShipVia': 3, 'Freight': 146.06, 'ShipName': 'Ernst Handel', 'ShipAddress': 'Kirchgasse 6', 'ShipCity': 'Graz', 'ShipRegion': None, 'ShipPostalCode': '8010', 'ShipCountry': 'Austria'}, {'OrderID': 10351, 'CustomerID': 'ERNSH', 'EmployeeID': 1, 'OrderDate': '1996-11-11T00:00:00Z', 'RequiredDate': '1996-12-09T00:00:00Z', 'ShippedDate': '1996-11-20T00:00:00Z', 'S

<h2>use three parameters - for field type, operator , value